# Modélisation du prix des logements (DVF 2025)

**Objectif :** prédire le prix d'une maison ou d'un appartement à partir de ses caractéristiques.

**Données :** le jeu préparé dans le notebook d'exploration (une ligne par vente d'un seul logement, valeurs aberrantes retirées).

**Cible :** le logarithme de la valeur foncière (log_valeur), choisi car c'est la distribution la plus symétrique.

**Plan :**
1. Chargement et contrôle des données
2. Feature engineering
3. Séparation train / test
4. Baseline
5. Régression linéaire, Ridge et Lasso
6. Arbre de décision et Random Forest
7. Comparaison des modèles

## 1. Chargement et contrôle des données

In [9]:
import pandas as pd
import numpy as np
import plotly.express as px

pd.set_option("display.max_columns", None)

In [10]:
filepath="clean_dvf_2025_ventes.parquet"
ventes = pd.read_parquet(filepath)
ventes.shape

(806184, 17)

In [11]:
ventes.head()

,date,nature_mutation,valeur_fonciere,code_insee,code_departement,nb_lignes,nb_appartement,nb_dépendance,nb_local,nb_maison,surface_bati,nb_pieces,surface_terrain,type_vente,prix_m2,doublon_ambigu,log_valeur
id_mutation,,,,,,,,,,,,,,,,,
0,2025-01-07,Vente,468000.0,01158,01,3,0.0,1.0,0.0,1.0,111.0,5.0,211.0,Maison seule,4216.216216,False,13.056224
1,2025-01-06,Vente,180000.0,01257,01,3,1.0,0.0,0.0,0.0,87.0,4.0,139.0,Appartement seul,2068.965517,False,12.100712
2,2025-01-06,Vente,490000.0,01065,01,12,0.0,1.0,0.0,1.0,190.0,6.0,3637.0,Maison seule,2578.947368,False,13.102161
3,2025-01-09,Vente,183000.0,01173,01,3,1.0,2.0,0.0,0.0,35.0,2.0,0.0,Appartement seul,5228.571429,False,12.117241
4,2025-01-07,Vente,295000.0,01028,01,3,0.0,0.0,0.0,1.0,95.0,2.0,1305.0,Maison seule,3105.263158,False,12.594731


In [12]:
ventes.dtypes

,0
date,datetime64[ns]
nature_mutation,category
valeur_fonciere,float64
code_insee,object
code_departement,category
nb_lignes,int64
nb_appartement,float64
nb_dépendance,float64
nb_local,float64
nb_maison,float64


In [13]:
ventes.isna().sum()

,0
date,0
nature_mutation,0
valeur_fonciere,0
code_insee,0
code_departement,0
nb_lignes,0
nb_appartement,0
nb_dépendance,0
nb_local,0
nb_maison,0


## 2. Feature engineering
### 2.1 Vérification des colonnes candidates à la suppression

In [14]:
# Si une colonne n'a qu'une seule valeur, elle n'apporte aucune information au modèle
print(ventes["nb_local"].value_counts())
print(ventes["doublon_ambigu"].value_counts())
print(ventes["nature_mutation"].value_counts())

nb_local
0.0    806184
Name: count, dtype: int64
doublon_ambigu
False    806184
Name: count, dtype: int64
nature_mutation
Vente                                 798636
Vente en l'état futur d'achèvement      7548
Echange                                    0
Adjudication                               0
Expropriation                              0
Vente terrain à bâtir                      0
Name: count, dtype: int64


**Constat :** nb_local (toujours 0) et doublon_ambigu (toujours False) sont constants, car le périmètre ne garde que les ventes sans local commercial et sans doublon ambigu : ces deux colonnes seront retirées. Les VEFA ne représentent que 7 548 ventes (0,9 %).

### 2.2 Création de nouvelles variables

In [15]:

ventes["est_vefa"] = np.where(ventes["nature_mutation"] == "Vente en l'état futur d'achèvement", 1, 0)
ventes["a_dependance"] = np.where(ventes["nb_dépendance"] > 0, 1, 0)

ventes[["est_vefa", "a_dependance"]].mean()

,0
est_vefa,0.009363
a_dependance,0.560556


**Constat :** 0,9 % des ventes sont des VEFA et 56 % comprennent au moins une dépendance (cave, garage, parking).

In [16]:
# Le mois seul (1 à 12) ferait croire au modèle que décembre (12) est très loin de janvier (1).
# Avec sin et cos, les mois sont placés sur un cercle : décembre et janvier deviennent voisins.
ventes["mois"] = ventes["date"].dt.month
ventes["mois_sin"] = np.sin(2 * np.pi * ventes["mois"] / 12)
ventes["mois_cos"] = np.cos(2 * np.pi * ventes["mois"] / 12)

ventes[["mois", "mois_sin", "mois_cos"]].drop_duplicates().sort_values("mois")

,mois,mois_sin,mois_cos
id_mutation,,,
0,1,5.000000e-01,8.660254e-01
552,2,8.660254e-01,5.000000e-01
1579,3,1.000000e+00,6.123234e-17
2553,4,8.660254e-01,-5.000000e-01
3968,5,5.000000e-01,-8.660254e-01
4888,6,1.224647e-16,-1.000000e+00
6020,7,-5.000000e-01,-8.660254e-01
7531,8,-8.660254e-01,-5.000000e-01
8417,9,-1.000000e+00,-1.836970e-16


**Constat :** chaque mois a une position unique sur le cercle (sin, cos), et décembre (12) est voisin de janvier (1).

In [17]:
# Nombre de ventes par mois
px.bar(ventes["mois"].value_counts().sort_index(), title="Nombre de ventes par mois")

**Constat :** toute l'année 2025 est couverte. Le nombre de ventes varie d'environ 53 000 en mai à 89 000 en juillet, avec un creux en août.

In [18]:
# Prix médian au m² par mois pour vérifier si le prix varie selon la saison
ventes.groupby("mois")["prix_m2"].median().round()

,prix_m2
mois,
1,2576.0
2,2556.0
3,2626.0
4,2519.0
5,2542.0
6,2642.0
7,2643.0
8,2551.0
9,2664.0


**Constat :** le prix médian au m² varie peu selon le mois, de 2 519 €/m² en avril à 2 664 €/m² en septembre (environ 6 %), alors qu'il est 11,4 fois plus élevé à Paris que dans la Creuse.

## 3. Séparation train / test
Les features sont choisies en excluant valeur_fonciere et prix_m2, qui servent à calculer la cible (data leakage). Le test sert à évaluer le modèle sur des ventes qu'il n'a jamais vues.

In [19]:
features_num = ["surface_bati", "nb_pieces", "surface_terrain", "est_vefa", "a_dependance", "mois_sin", "mois_cos"]
features_cat = ["type_vente", "code_departement"]

# On garde code_insee et prix_m2 de côté : ils serviront à calculer le prix médian de la commune
# (sur le train uniquement), mais ils ne seront pas donnés tels quels au modèle
X = ventes[features_num + features_cat + ["code_insee", "prix_m2"]]
y = ventes["log_valeur"]

X.shape, y.shape

((806184, 11), (806184,))

In [20]:
from sklearn.model_selection import train_test_split

# random_state fixe le tirage : on obtient la même séparation à chaque exécution (reproductibilité)
# test_size=0.25 : 25 % des ventes servent au test, comme la valeur par défaut vue en cours
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=41)

print("Train :", X_train.shape)
print("Test :", X_test.shape)

Train : (604638, 11)
Test : (201546, 11)


In [21]:
# la cible doit avoir à peu près la même distribution dans le train et dans le test
pd.DataFrame({"train": y_train.describe(), "test": y_test.describe()})

,train,test
count,604638.000000,201546.000000
mean,12.120970,12.121546
std,0.725786,0.727821
min,6.401054,6.306275
25%,11.695247,11.695247
50%,12.144197,12.141001
75%,12.573317,12.574182
max,16.506754,16.835935


**Constat :** 604 638 ventes en train (75 %) et 201 546 en test (25 %). La cible a la même distribution dans les deux jeux (moyenne de 12,12 et écart-type de 0,73) : la séparation aléatoire n'a pas créé de déséquilibre.

### 3.2 Prix médian au m² de la commune
La commune est remplacée par son prix médian au m², calculé sur le train uniquement pour éviter le data leakage. Les communes de moins de 10 ventes dans le train, ou absentes du train, reçoivent le prix médian de leur département.

In [22]:
MIN_VENTES = 10

# Médiane et nombre de ventes de chaque commune, calculés sur le train uniquement
stats_commune = X_train.groupby("code_insee")["prix_m2"].agg(["median", "count"])

# On ne garde que les communes avec assez de ventes pour que la médiane soit fiable
mediane_commune = stats_commune[stats_commune["count"] >= MIN_VENTES]["median"]

# Médiane de chaque département, sur le train. on utilise  astype(str) pour éviter les problèmes du type category avec map
mediane_dep = X_train.groupby(X_train["code_departement"].astype(str))["prix_m2"].median()

print("Communes dans le train :", len(stats_commune))
print(f"Communes avec au moins {MIN_VENTES} ventes :", len(mediane_commune))

Communes dans le train : 29237
Communes avec au moins 10 ventes : 9011


In [23]:
def ajouter_prix_commune(X):
    # copy() pour éviter de modifier X_train ou X_test sans le vouloir
    X = X.copy()
    X["code_departement"] = X["code_departement"].astype(str)

    # map remplace chaque code commune par sa médiane ; les communes absentes deviennent NaN
    X["prix_med_commune"] = X["code_insee"].map(mediane_commune)

    # Les NaN sont remplacés par la médiane du département
    X["prix_med_commune"] = X["prix_med_commune"].fillna(X["code_departement"].map(mediane_dep))

    # prix_m2 et code_insee ont servi au calcul : on les retire pour qu'ils n'entrent pas dans le modèle
    return X.drop(columns=["prix_m2", "code_insee"])


X_train = ajouter_prix_commune(X_train)
X_test = ajouter_prix_commune(X_test)

X_train.head()

,surface_bati,nb_pieces,surface_terrain,est_vefa,a_dependance,mois_sin,mois_cos,type_vente,code_departement,prix_med_commune
id_mutation,,,,,,,,,,
142267,30.0,1.0,0.0,0,0,-2.449294e-16,1.000000e+00,Appartement seul,13,5284.437500
1185792,162.0,6.0,1057.0,0,1,5.000000e-01,-8.660254e-01,Maison seule,89,1746.031746
274474,64.0,3.0,0.0,0,1,5.000000e-01,8.660254e-01,Appartement seul,25,2238.297872
541532,17.0,1.0,0.0,0,1,5.000000e-01,-8.660254e-01,Maison seule,40,2091.127013
1108099,23.0,1.0,0.0,0,0,1.000000e+00,6.123234e-17,Appartement seul,83,6081.268959


In [24]:
# isin vérifie, pour chaque vente du test, si sa commune fait partie des communes à au moins 10 ventes
commune_connue = ventes.loc[X_test.index, "code_insee"].isin(mediane_commune.index)
print(f"Ventes du test avec la médiane du département : {1 - commune_connue.mean():.1%}")

print("Valeurs manquantes :", X_train["prix_med_commune"].isna().sum(), "/", X_test["prix_med_commune"].isna().sum())

Ventes du test avec la médiane du département : 13.3%
Valeurs manquantes : 0 / 0


**Constat :** sur les 29 237 communes du train, seules 9 011 (31 %) ont au moins 10 ventes. Mais comme les ventes sont concentrées dans les communes les plus actives, 86,7 % des ventes du test reçoivent la médiane de leur commune, et 13,3 % celle de leur département. Aucune valeur manquante ne reste.

## 4. Baseline
Deux modèles de référence, que les modèles de machine learning devront battre :
1. la moyenne de la cible du train, prédite pour toutes les ventes (R² = 0 par définition) ;
2. une règle simple : prix médian au m² de la commune × surface.

Les erreurs sont calculées en euros (après np.exp) pour être compréhensibles. Le R² est calculé sur le logarithme, qui est la cible apprise par les modèles.

In [25]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


def evaluer(nom, y_vrai_log, y_pred_log):
    # np.exp annule le logarithme : on revient à des prix en euros
    y_vrai = np.exp(y_vrai_log)
    y_pred = np.exp(y_pred_log)

    return {
        "modele": nom,
        "MAE (€)": round(mean_absolute_error(y_vrai, y_pred)),
        # La RMSE est la racine de la MSE : elle pénalise davantage les grosses erreurs
        "RMSE (€)": round(np.sqrt(mean_squared_error(y_vrai, y_pred))),
        "R² (log)": round(r2_score(y_vrai_log, y_pred_log), 3),
    }


# Liste qui accumulera les résultats de tous les modèles, pour les comparer à la fin
resultats = []

In [26]:
# Baseline 1 : la moyenne du train, répétée autant de fois qu'il y a de ventes dans le test
pred_moyenne = np.full(len(y_test), y_train.mean())
resultats.append(evaluer("Baseline : moyenne", y_test, pred_moyenne))

# Baseline 2 : prix médian au m² de la commune × surface, puis log pour être comparable à la cible
pred_regle = np.log(X_test["prix_med_commune"] * X_test["surface_bati"])
resultats.append(evaluer("Baseline : médiane commune × surface", y_test, pred_regle))

pd.DataFrame(resultats)

,modele,MAE (€),RMSE (€),R² (log)
0,Baseline : moyenne,123417,237398,-0.000
1,Baseline : médiane commune × surface,63177,119928,0.671


**Constat :** la moyenne se trompe en moyenne de 123 417 € (R² = 0, prédicteur naïf). La règle « prix médian au m² de la commune × surface » divise l'erreur par deux : 63 177 € de MAE et un R² de 0,671. Sa RMSE (119 928 €) vaut presque le double de sa MAE : certaines ventes ont des erreurs très importantes.

**Objectif :** les modèles de machine learning doivent faire mieux que cette règle, soit une MAE inférieure à 63 177 € et un R² supérieur à 0,671 sur le test.

## 5. Régression linéaire


#5.1 avec les valeurs brutes

In [27]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression

# Les features numériques, avec la nouvelle colonne prix_med_commune
colonnes_num = features_num + ["prix_med_commune"]
colonnes_cat = features_cat

transfo_num = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("standard", StandardScaler()),
])

# drop="first" pour retirer une colonne par variable : elle se déduit des autres
transfo_cat = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(drop="first")),
])

preparation = ColumnTransformer([
    ("quantitative", transfo_num, colonnes_num),
    ("qualitative", transfo_cat, colonnes_cat),
])

In [28]:
modele_lr = Pipeline([
    ("prep", preparation),
    ("reg", LinearRegression()),
])

# fit apprend en une fois : les moyennes et écarts-types du scaler, les colonnes du one-hot, puis les coefficients
modele_lr.fit(X_train, y_train)

# On évalue sur le train ET sur le test : un grand écart entre les deux signale de l'overfitting
resultats.append(evaluer("Régression linéaire (train)", y_train, modele_lr.predict(X_train)))
resultats.append(evaluer("Régression linéaire (test)", y_test, modele_lr.predict(X_test)))

pd.DataFrame(resultats)

,modele,MAE (€),RMSE (€),R² (log)
0,Baseline : moyenne,123417,237398,-0.000
1,Baseline : médiane commune × surface,63177,119928,0.671
2,Régression linéaire (train),75461,452909,0.633
3,Régression linéaire (test),77667,608730,0.633


In [29]:
# Les coefficients de la régression : c'est ce qui la rend interprétable
noms = modele_lr.named_steps["prep"].get_feature_names_out()
coefs = pd.Series(modele_lr.named_steps["reg"].coef_, index=noms)

# On affiche les variables numériques et le type de bien ; les ~100 départements sont trop nombreux pour être lus ici
coefs[~coefs.index.str.contains("code_departement")].sort_values()

,0
quantitative__mois_sin,-0.011694
quantitative__mois_cos,-0.006688
quantitative__surface_terrain,0.003540
quantitative__est_vefa,0.040983
quantitative__a_dependance,0.042778
quantitative__nb_pieces,0.109250
qualitative__type_vente_Maison seule,0.176270
quantitative__surface_bati,0.310349
quantitative__prix_med_commune,0.491288


**Constat :** la régression linéaire fait moins bien que la règle de la baseline (MAE de 77 667 € contre 63 177 €, R² de 0,633 contre 0,671). Le R² est identique sur le train et le test : il n'y a pas d'overfitting, c'est au contraire un modèle pas assez variant. La RMSE (608 716 €) vaut 8 fois la MAE : quelques prédictions sont très fausses.

**Coefficients :** le prix médian de la commune (0,49) et la surface (0,31) sont les variables les plus importantes, suivies du type de bien (0,18 pour une maison) et du nombre de pièces (0,11). Le mois et la surface du terrain ne pèsent presque rien.

**Explication :** le prix est le produit d'un prix au m² et d'une surface. Le logarithme transforme ce produit en somme, log(prix) = log(prix au m²) + log(surface), ce qu'une régression linéaire peut apprendre. Mais seulement si on lui donne les logarithmes de ces variables, et non leurs valeurs brutes.

In [30]:
# Les prédictions les plus élevées, en euros
pred_test = np.exp(modele_lr.predict(X_test))
print("Prédiction maximale :", round(pred_test.max()), "€")
print("Prix réel maximal :", round(np.exp(y_test).max()), "€")

Prédiction maximale : 157529222 €
Prix réel maximal : 20500000 €


In [31]:
# argmax pour trouver la position de la prédiction la plus élevée ; iloc permet de retrouver la vente correspondante
position = pred_test.argmax()
X_test.iloc[position][["surface_bati", "nb_pieces", "prix_med_commune", "type_vente", "code_departement"]]

,956507
surface_bati,547.0
nb_pieces,9.0
prix_med_commune,13107.142857
type_vente,Appartement seul
code_departement,73


**Constat :** la prédiction la plus élevée concerne un appartement de 547 m² dans une commune à 13 107 €/m² (Savoie). La surface et le prix médian de la commune, les deux variables les plus importantes du modèle, sont toutes les deux extrêmes : additionnées en ligne droite dans le logarithme, puis repassées en euros avec np.exp, elles donnent 157,5 M€. La règle de la baseline donnerait environ 7,2 M€ pour cette vente.

### 5.2 Régression linéaire avec des variables en logarithme
La surface du terrain et le prix médian de la commune sont passés au logarithme, pour que la relation avec la cible devienne linéaire.

In [32]:
for X_ in [X_train, X_test]:
    X_["log_surface_bati"] = np.log(X_["surface_bati"])
    X_["log_prix_med_commune"] = np.log(X_["prix_med_commune"])
    X_["log_surface_terrain"] = np.log1p(X_["surface_terrain"])

colonnes_num_log = ["log_surface_bati", "log_prix_med_commune", "log_surface_terrain",
                    "nb_pieces", "est_vefa", "a_dependance", "mois_sin", "mois_cos"]

# Même préparation qu'avant, avec la nouvelle liste de colonnes numériques
preparation_log = ColumnTransformer([
    ("quantitative", transfo_num, colonnes_num_log),
    ("qualitative", transfo_cat, colonnes_cat),
])

In [33]:
modele_lr_log = Pipeline([
    ("prep", preparation_log),
    ("reg", LinearRegression()),
])

modele_lr_log.fit(X_train, y_train)

resultats.append(evaluer("Régression linéaire log (train)", y_train, modele_lr_log.predict(X_train)))
resultats.append(evaluer("Régression linéaire log (test)", y_test, modele_lr_log.predict(X_test)))

pd.DataFrame(resultats)

,modele,MAE (€),RMSE (€),R² (log)
0,Baseline : moyenne,123417,237398,-0.000
1,Baseline : médiane commune × surface,63177,119928,0.671
2,Régression linéaire (train),75461,452909,0.633
3,Régression linéaire (test),77667,608730,0.633
4,Régression linéaire log (train),62154,124624,0.698
5,Régression linéaire log (test),63031,126727,0.692


In [34]:
noms = modele_lr_log.named_steps["prep"].get_feature_names_out()
coefs_log = pd.Series(modele_lr_log.named_steps["reg"].coef_, index=noms)
coefs_log[~coefs_log.index.str.contains("code_departement")].sort_values()

,0
quantitative__mois_sin,-0.011329
quantitative__mois_cos,-0.005405
qualitative__type_vente_Maison seule,0.009180
quantitative__nb_pieces,0.023779
quantitative__a_dependance,0.025535
quantitative__est_vefa,0.032972
quantitative__log_surface_terrain,0.067262
quantitative__log_surface_bati,0.429183
quantitative__log_prix_med_commune,0.526520


**Constat :** avec les variables en logarithme, les prédictions aberrantes disparaissent : la RMSE est divisée par presque 5 (126 727 € contre 608 716 €). La régression bat la baseline en MAE (63 031 € contre 63 177 €) et en R² (0,692 contre 0,671), mais le gain est faible, et sa RMSE reste un peu moins bonne que celle de la règle. Le modèle retrouve surtout la règle « prix médian de la commune × surface ». Les scores sont proches sur le train et le test donc il n'y a pas d'overfitting.

**Coefficients :** le prix médian de la commune (0,53) et la surface (0,43) restent les variables principales. Le coefficient du type « maison » chute de 0,18 à 0,009 : il compensait probablement, dans le premier modèle, la mauvaise prise en compte de la surface (hypothèse).

### 5.3 Ridge et Lasso
Nous allons utiliser Ridge et Lasso pour ajouter une pénalité sur la taille des coefficients (L2 pour Ridge, L1 pour Lasso). alpha va règler la force de cette pénalité, et il sera choisi par validation croisée avec GridSearchCV. Comme la régression linéaire n'overfit pas, on s'attend à un gain faible.

In [35]:
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import GridSearchCV

modele_ridge = Pipeline([
    ("prep", preparation_log),
    ("reg", Ridge()),
])

# "reg__alpha" : le double tiret désigne le paramètre alpha de l'étape "reg" du pipeline
# cv=5 : chaque valeur d'alpha est testée sur 5 découpages différents du train
grid_ridge = GridSearchCV(modele_ridge, param_grid={"reg__alpha": [0.01, 0.1, 1, 10, 100]}, cv=5)
grid_ridge.fit(X_train, y_train)

print("Meilleur alpha :", grid_ridge.best_params_)

Meilleur alpha : {'reg__alpha': 100}


In [36]:
modele_lasso = Pipeline([
    ("prep", preparation_log),
    ("reg", Lasso()),
])

# Le Lasso met des coefficients à 0 : un alpha trop grand les supprimerait tous, d'où des valeurs plus petites
grid_lasso = GridSearchCV(modele_lasso, param_grid={"reg__alpha": [0.0001, 0.001, 0.01, 0.1]}, cv=5)
grid_lasso.fit(X_train, y_train)

print("Meilleur alpha :", grid_lasso.best_params_)

Meilleur alpha : {'reg__alpha': 0.0001}


In [37]:
# best_estimator_ est le pipeline réentraîné sur tout le train avec le meilleur alpha
for nom, grid in [("Ridge", grid_ridge), ("Lasso", grid_lasso)]:
    resultats.append(evaluer(f"{nom} (train)", y_train, grid.best_estimator_.predict(X_train)))
    resultats.append(evaluer(f"{nom} (test)", y_test, grid.best_estimator_.predict(X_test)))

pd.DataFrame(resultats)

,modele,MAE (€),RMSE (€),R² (log)
0,Baseline : moyenne,123417,237398,-0.000
1,Baseline : médiane commune × surface,63177,119928,0.671
2,Régression linéaire (train),75461,452909,0.633
3,Régression linéaire (test),77667,608730,0.633
4,Régression linéaire log (train),62154,124624,0.698
5,Régression linéaire log (test),63031,126727,0.692
6,Ridge (train),62152,124603,0.698
7,Ridge (test),63030,126703,0.692
8,Lasso (train),62100,124360,0.698
9,Lasso (test),62987,126427,0.691


In [38]:
# Combien de coefficients le Lasso a-t-il mis à 0 ?
coefs_lasso = pd.Series(grid_lasso.best_estimator_.named_steps["reg"].coef_,
                        index=grid_lasso.best_estimator_.named_steps["prep"].get_feature_names_out())

print("Coefficients mis à 0 :", (coefs_lasso == 0).sum(), "sur", len(coefs_lasso))
coefs_lasso[coefs_lasso == 0]

Coefficients mis à 0 : 30 sur 105


,0
qualitative__code_departement_04,0.0
qualitative__code_departement_05,-0.0
qualitative__code_departement_07,0.0
qualitative__code_departement_08,0.0
qualitative__code_departement_09,-0.0
qualitative__code_departement_10,-0.0
qualitative__code_departement_14,-0.0
qualitative__code_departement_19,-0.0
qualitative__code_departement_24,-0.0
qualitative__code_departement_28,0.0


**Constat :** Ridge (meilleur alpha : 100) et Lasso (meilleur alpha : 0,0001) donnent les mêmes résultats que la régression linéaire, à quelques dizaines d'euros près (MAE de 63 030 € et 62 987 € sur le test). C'est attendu car la régularisation réduit l'overfitting, et la régression linéaire n'en avait pas. Le Lasso choisit la pénalité la plus faible de la grille, ce qui confirme qu'elle n'aide pas. Le Ridge choisit la plus forte, mais avec 600 000 ventes, une pénalité de 100 reste très faible face à l'erreur cumulée sur toutes les ventes (hypothèse) : ses coefficients changent à peine. Comme cette valeur est au bord de la grille, des alphas plus grands n'ont pas été testés.

**Lasso :** 30 coefficients sur 105 sont mis à zéro, uniquement des départements. Hypothèse : le prix médian de la commune contient déjà l'information de localisation, ce qui rend l'indicatrice de ces départements inutile.

## 6. Arbre de décision et Random Forest


### 6.1 Arbre de décision limité
# max_depth limite le nombre de questions successives, et min_samples_leaf impose un nombre minimum de ventes par feuille. Les deux empêchent l'arbre d'apprendre des cas isolés.

In [39]:
from sklearn.tree import DecisionTreeRegressor

# Pour les arbres : sparse_output=False renvoie un tableau classique, beaucoup plus rapide à découper
transfo_cat_arbre = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(drop="first", sparse_output=False)),
])

preparation_arbre = ColumnTransformer([
    ("quantitative", transfo_num, colonnes_num_log),
    ("qualitative", transfo_cat_arbre, colonnes_cat),
])

In [40]:
modele_arbre = Pipeline([
    ("prep", preparation_arbre),
    # random_state rend l'arbre reproductible quand plusieurs découpages sont aussi bons
    ("reg", DecisionTreeRegressor(random_state=41)),
])

grille_arbre = {
    "reg__max_depth": [10, 15, 20],
    "reg__min_samples_leaf": [10, 50, 200],
}

# cv=3 : 9 combinaisons × 3 découpages = 27 arbres ; n_jobs=-1 utilise tous les processeurs
grid_arbre = GridSearchCV(modele_arbre, param_grid=grille_arbre, cv=3, n_jobs=-1)
grid_arbre.fit(X_train, y_train)

print("Meilleurs paramètres :", grid_arbre.best_params_)

resultats.append(evaluer("Arbre de décision (train)", y_train, grid_arbre.best_estimator_.predict(X_train)))
resultats.append(evaluer("Arbre de décision (test)", y_test, grid_arbre.best_estimator_.predict(X_test)))

pd.DataFrame(resultats)

Meilleurs paramètres : {'reg__max_depth': 15, 'reg__min_samples_leaf': 50}


,modele,MAE (€),RMSE (€),R² (log)
0,Baseline : moyenne,123417,237398,-0.000
1,Baseline : médiane commune × surface,63177,119928,0.671
2,Régression linéaire (train),75461,452909,0.633
3,Régression linéaire (test),77667,608730,0.633
4,Régression linéaire log (train),62154,124624,0.698
5,Régression linéaire log (test),63031,126727,0.692
6,Ridge (train),62152,124603,0.698
7,Ridge (test),63030,126703,0.692
8,Lasso (train),62100,124360,0.698
9,Lasso (test),62987,126427,0.691


**Constat :** l'arbre de décision (profondeur 15, au moins 50 ventes par feuille) est le premier modèle qui bat nettement la baseline : MAE de 60 664 € contre 63 177 € (−4 %), R² de 0,707 contre 0,671. Sa RMSE reste au niveau de celle de la baseline : il réduit l'erreur moyenne, mais pas les plus grosses erreurs. L'écart entre le train (0,732) et le test (0,707) montre un léger overfitting, plus marqué que pour les modèles linéaires.

In [41]:
# Score de chaque combinaison testée pour voir l'effet de la profondeur et de la taille des feuilles
scores = pd.DataFrame(grid_arbre.cv_results_)[["param_reg__max_depth", "param_reg__min_samples_leaf", "mean_test_score"]]
scores.sort_values("mean_test_score", ascending=False)

,param_reg__max_depth,param_reg__min_samples_leaf,mean_test_score
4,15,50,0.707220
7,20,50,0.705992
5,15,200,0.704026
8,20,200,0.704013
1,10,50,0.703532
0,10,10,0.703277
2,10,200,0.701283
3,15,10,0.697864
6,20,10,0.681304


**Constat :** les arbres avec des feuilles de 10 ventes ont les moins bons scores, surtout quand ils sont profonds (0,681 pour une profondeur de 20) : ils apprennent des cas particuliers du train, c'est de l'overfitting. Avec des feuilles de 200 ventes, la profondeur ne change presque plus rien. Le meilleur compromis est une profondeur de 15 et des feuilles d'au moins 50 ventes.

## Conclusion

L'arbre de décision (profondeur 15, au moins 50 ventes par feuille) est le meilleur modèle : il se trompe en moyenne de 60 664 € sur le test, avec un R² de 0,707. C'est le seul qui bat nettement la règle simple « prix médian au m² de la commune × surface » mais le gain reste modeste (−4 %). La régression linéaire n'égale cette règle qu'avec les variables en logarithme, et Ridge et Lasso n'apportent presque rien, car elle n'overfit pas. La localisation et la surface sont, de loin, les variables les plus importantes.

Le modèle donne donc un ordre de grandeur du prix, pas un prix précis et aucun modèle ne réduit les plus grosses erreurs. Ces résultats restent limités par les données de la DVF qui ne décrit ni l'état du logement, ni l'étage et par nos choix de nettoyage.
Entrainer une randomForest et avoir une localisation plus fine (latitude et longitude) des biens seraient les prochaines pistes.